# FloraScan - Model Comparison Dashboard
Comparing all explored architectures on PlantVillage dataset.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs(r'd:\Florascann\results', exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.facecolor'] = 'white'

## Model Exploration Results

In [ ]:
# Actual training results from all notebooks
models = {
    'ANN Baseline': {
        'notebook': '00_ann_baseline.ipynb',
        'train_acc': 7.02,
        'val_acc': 7.52,
        'description': 'Simple Dense layers only',
        'version': 'v0'
    },
    'Basic CNN': {
        'notebook': '01_basic_cnn.ipynb',
        'train_acc': 100.0,
        'val_acc': 68.67,
        'description': '3-layer CNN, no regularization',
        'version': 'v1'
    },
    'Improved CNN': {
        'notebook': '02_improved_cnn.ipynb',
        'train_acc': 85.64,
        'val_acc': 80.44,
        'description': 'CNN + BatchNorm + Dropout + Augmentation',
        'version': 'v2'
    },
    'Transfer Learning': {
        'notebook': '03_transfer_learning.ipynb',
        'train_acc': 80.77,
        'val_acc': 83.45,
        'description': 'MobileNetV2 (frozen) + custom head',
        'version': 'v3'
    },
    'Fine-tuned (Retrained)': {
        'notebook': '05_retrain_model.ipynb',
        'train_acc': 82.17,
        'val_acc': 84.54,
        'description': 'MobileNetV2 fine-tuned (best model)',
        'version': 'v5'
    }
}

model_names = list(models.keys())
train_accs = [models[m]['train_acc'] for m in model_names]
val_accs = [models[m]['val_acc'] for m in model_names]

print("="*70)
print("           FLORASCAN - MODEL EXPLORATION SUMMARY")
print("="*70)
for name, data in models.items():
    print(f"\n{data['version']}: {name}")
    print(f"   Train: {data['train_acc']:.1f}% | Val: {data['val_acc']:.1f}%")
    print(f"   {data['description']}")
print("\n" + "="*70)

## 1. Train vs Validation Accuracy

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(len(model_names))
width = 0.35

colors_train = ['#3498db', '#3498db', '#3498db', '#3498db', '#2ecc71']
colors_val = ['#e74c3c', '#f39c12', '#27ae60', '#27ae60', '#27ae60']

bars1 = ax.bar(x - width/2, train_accs, width, label='Training Accuracy', 
               color='#3498db', alpha=0.85, edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, val_accs, width, label='Validation Accuracy', 
               color='#2ecc71', alpha=0.85, edgecolor='black', linewidth=0.5)

ax.set_xlabel('Model Architecture', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=13, fontweight='bold')
ax.set_title('FloraScan: Model Performance Comparison', fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(['ANN', 'Basic\nCNN', 'Improved\nCNN', 'Transfer\nLearning', 'Fine-tuned\n(Best)'], fontsize=11)
ax.legend(loc='upper left', fontsize=11)
ax.set_ylim(0, 115)

for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=10)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=10, 
                fontweight='bold', color='#27ae60')

# Highlight best model
ax.axhline(y=84.54, color='green', linestyle='--', alpha=0.5, linewidth=2)
ax.text(4.5, 86, 'Best: 84.5%', fontsize=10, color='green', fontweight='bold')

plt.tight_layout()
plt.savefig(r'd:\Florascann\results\model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: model_comparison.png")

## 2. Accuracy Progression

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

x = range(1, len(model_names) + 1)

ax.plot(x, val_accs, 'o-', linewidth=3, markersize=14, color='#2ecc71', zorder=5, label='Validation')
ax.plot(x, train_accs, 's--', linewidth=2, markersize=10, color='#3498db', alpha=0.6, label='Training')
ax.fill_between(x, val_accs, alpha=0.2, color='#2ecc71')

for i, (xi, yi) in enumerate(zip(x, val_accs)):
    color = '#e74c3c' if yi < 40 else '#f39c12' if yi < 70 else '#27ae60'
    ax.annotate(f'{yi:.1f}%', xy=(xi, yi), xytext=(0, 18), textcoords='offset points',
                ha='center', fontsize=12, fontweight='bold', color=color,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.9))

ax.set_xlabel('Model Evolution', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy (%)', fontsize=13, fontweight='bold')
ax.set_title('FloraScan: Accuracy Improvement Journey', fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(['ANN\n(v0)', 'Basic CNN\n(v1)', 'Improved CNN\n(v2)', 'Transfer Learning\n(v3)', 'Fine-tuned\n(v5)'], fontsize=10)
ax.set_ylim(0, 105)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

improvement = val_accs[-1] - val_accs[0]
ax.annotate(f'+{improvement:.1f}% total improvement!', xy=(3, 30), fontsize=14, 
            fontweight='bold', color='#8e44ad', ha='center',
            bbox=dict(boxstyle='round', facecolor='#f5f5f5', edgecolor='#8e44ad'))

plt.tight_layout()
plt.savefig(r'd:\Florascann\results\accuracy_progression.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: accuracy_progression.png")

## 3. Overfitting Analysis

In [ ]:
gaps = [t - v for t, v in zip(train_accs, val_accs)]

fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#e74c3c' if g > 25 else '#f39c12' if g > 10 else '#2ecc71' for g in gaps]
bars = ax.bar(range(len(model_names)), gaps, color=colors, edgecolor='black', linewidth=1.2)

ax.set_xlabel('Model Architecture', fontsize=12, fontweight='bold')
ax.set_ylabel('Overfitting Gap (Train - Val) %', fontsize=12, fontweight='bold')
ax.set_title('FloraScan: Overfitting Analysis', fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(range(len(model_names)))
ax.set_xticklabels(['ANN', 'Basic CNN', 'Improved CNN', 'Transfer\nLearning', 'Fine-tuned'], fontsize=10)

ax.axhline(y=10, color='orange', linestyle='--', linewidth=2, label='Moderate (<10%)')
ax.axhline(y=5, color='green', linestyle='--', linewidth=2, label='Good (<5%)')
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)

for bar, gap in zip(bars, gaps):
    label = f'{gap:.1f}%'
    ypos = gap + 1 if gap >= 0 else gap - 3
    ax.annotate(label, xy=(bar.get_x() + bar.get_width()/2, ypos),
                ha='center', fontsize=11, fontweight='bold')

ax.legend(loc='upper right')
ax.set_ylim(-10, max(gaps) + 10)

plt.tight_layout()
plt.savefig(r'd:\Florascann\results\overfitting_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: overfitting_analysis.png")

## 4. Complete Dashboard

In [ ]:
fig = plt.figure(figsize=(16, 12))
fig.suptitle('FloraScan: Model Exploration Dashboard', fontsize=20, fontweight='bold', y=0.98)

gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)

# Plot 1: Bar Chart
ax1 = fig.add_subplot(gs[0, :2])
x = np.arange(len(model_names))
width = 0.35
ax1.bar(x - width/2, train_accs, width, label='Train', color='#3498db', alpha=0.8)
ax1.bar(x + width/2, val_accs, width, label='Validation', color='#2ecc71', alpha=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(['ANN', 'Basic\nCNN', 'Improved\nCNN', 'Transfer\nLearning', 'Fine-tuned'], fontsize=9)
ax1.set_ylabel('Accuracy (%)')
ax1.set_title('Train vs Validation Accuracy', fontweight='bold', fontsize=12)
ax1.legend(loc='upper left')
ax1.set_ylim(0, 115)
for i, (t, v) in enumerate(zip(train_accs, val_accs)):
    ax1.text(i - width/2, t + 2, f'{t:.0f}', ha='center', fontsize=8)
    ax1.text(i + width/2, v + 2, f'{v:.0f}', ha='center', fontsize=8, fontweight='bold')

# Plot 2: Final Model Pie
ax2 = fig.add_subplot(gs[0, 2])
final_acc = val_accs[-1]
wedges, texts, autotexts = ax2.pie([final_acc, 100-final_acc], 
        labels=['Correct', 'Error'], 
        autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'],
        explode=(0.05, 0), startangle=90, 
        textprops={'fontsize': 11, 'fontweight': 'bold'})
ax2.set_title(f'Best Model Accuracy\n{final_acc:.1f}%', fontweight='bold', fontsize=12)

# Plot 3: Progression Line
ax3 = fig.add_subplot(gs[1, :2])
ax3.plot(range(1,6), val_accs, 'o-', linewidth=3, markersize=12, color='#2ecc71', label='Validation')
ax3.plot(range(1,6), train_accs, 's--', linewidth=2, markersize=8, color='#3498db', alpha=0.6, label='Training')
ax3.fill_between(range(1,6), val_accs, alpha=0.2, color='#2ecc71')
ax3.set_xticks(range(1,6))
ax3.set_xticklabels(['ANN', 'Basic\nCNN', 'Improved\nCNN', 'Transfer\nLearning', 'Fine-tuned'], fontsize=9)
ax3.set_ylabel('Accuracy (%)')
ax3.set_title('Accuracy Improvement Journey', fontweight='bold', fontsize=12)
ax3.set_ylim(0, 105)
ax3.legend(loc='lower right')
ax3.grid(True, alpha=0.3)
for i, v in enumerate(val_accs, 1):
    ax3.annotate(f'{v:.1f}%', xy=(i, v+4), ha='center', fontsize=9, fontweight='bold')

# Plot 4: Overfitting Bars
ax4 = fig.add_subplot(gs[1, 2])
colors = ['#2ecc71' if g < 5 else '#f39c12' if g < 15 else '#e74c3c' for g in gaps]
bars = ax4.barh(range(len(gaps)), gaps, color=colors)
ax4.set_yticks(range(len(gaps)))
ax4.set_yticklabels(['ANN', 'Basic', 'Improved', 'Transfer', 'Fine-tuned'], fontsize=9)
ax4.set_xlabel('Overfitting Gap (%)')
ax4.set_title('Overfitting Analysis', fontweight='bold', fontsize=12)
ax4.axvline(x=10, color='orange', linestyle='--', linewidth=1.5)
ax4.axvline(x=5, color='green', linestyle='--', linewidth=1.5)
for i, g in enumerate(gaps):
    ax4.text(max(g,0)+1, i, f'{g:.1f}%', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(r'd:\Florascann\results\model_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved: model_dashboard.png")

## Summary Table

In [ ]:
summary_data = []
for name, data in models.items():
    gap = data['train_acc'] - data['val_acc']
    status = 'SEVERE OVERFITTING' if gap > 25 else 'Moderate' if gap > 10 else 'Good' if gap > 0 else 'Excellent'
    summary_data.append({
        'Version': data['version'],
        'Model': name,
        'Train (%)': f"{data['train_acc']:.1f}",
        'Val (%)': f"{data['val_acc']:.1f}",
        'Gap (%)': f"{gap:.1f}",
        'Status': status
    })

df = pd.DataFrame(summary_data)
print("\n" + "="*90)
print("                       FLORASCAN - MODEL EXPLORATION RESULTS")
print("="*90)
print(df.to_string(index=False))
print("="*90)
print(f"\n BEST MODEL: Fine-tuned (Retrained) MobileNetV2")
print(f"   Validation Accuracy: {max(val_accs):.2f}%")
print(f"   Total Improvement: +{max(val_accs) - val_accs[0]:.1f}% from ANN baseline")
print(f"   Model file: models/model_v5_retrained.h5")

## Key Findings

| Model | Why We Tried It | What We Learned |
|-------|-----------------|------------------|
| **ANN (v0)** | Baseline comparison | Dense layers can't handle image spatial features (~7%) |
| **Basic CNN (v1)** | Add convolutions | CNNs work! But severe overfitting (100% train vs 69% val) |
| **Improved CNN (v2)** | Fix overfitting | BatchNorm + Dropout + Augmentation reduces gap to 5% |
| **Transfer Learning (v3)** | Use pretrained features | Pretrained MobileNetV2 boosts to 83.5% accuracy |
| **Fine-tuned (v5)** | Optimize further | Fine-tuning achieves best result: **84.5%** |

### Conclusion
Through systematic exploration, we improved accuracy from **7.5%** (ANN) to **84.5%** (Fine-tuned) - a **+77% improvement**!

The Fine-tuned MobileNetV2 model achieved the best results for plant disease detection with minimal overfitting.